# Reference Passing Test: Lightweight Reference Feature

This notebook demonstrates the lightweight reference passing feature for the Unified Reference System (URS). It shows how `get_data` and `write_data` methods can work with both value and reference modes using Pydantic Union types.


In [1]:
%pip install -qU "google-genai>=1.0.0" "pydantic>=2.0.0"


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


To run this notebook, your API key must be stored in a Colab Secret named `GOOGLE_API_KEY`. If you are running in a different environment, you can store your key in an environment variable.


In [ ]:
from google import genai
from google.colab import userdata
from typing import Optional, Union, Literal, Annotated
from pydantic import BaseModel, Field


client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))


/Users/oladipofasoro/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Define Pydantic Models

We define two Pydantic models that represent either a value or a reference. These will be used in Union types that translate to `anyOf` in the Google SDK tool definitions.


In [3]:
class ValueModel(BaseModel):
    """Represents a direct value (data passed directly)."""
    discr: Literal["value"] = "value"
    data: str = Field(..., description="The actual data value")

class ReferenceModel(BaseModel):
    """Represents a reference to data stored in the reference store."""
    discr: Literal["reference"] = "reference"
    ref_id: str = Field(..., description="The reference ID to retrieve data from the store")

# Union type for parameters and return values with discriminated union
ReferenceOrValue = Union[ValueModel, ReferenceModel]

# Request/Response wrapper models for get_data_ref
class GetDataRefRequest(BaseModel):
    """Request wrapper for get_data_ref function."""
    filename: ReferenceOrValue = Field(..., discriminator='discr', description="Either a ValueModel (direct filename) or ReferenceModel (reference to filename)")
    result_reference_id: Optional[str] = Field(None, description="If provided, stores the result as a reference and returns ReferenceModel. If None, returns ValueModel with the data.")

class GetDataRefResponse(BaseModel):
    """Response wrapper for get_data_ref function."""
    result: ReferenceOrValue = Field(..., discriminator='discr', description="Either a ValueModel (if result_reference_id was None) or ReferenceModel (if result_reference_id was provided)")

# Request/Response wrapper models for write_data_ref
class WriteDataRefRequest(BaseModel):
    """Request wrapper for write_data_ref function."""
    filename: ReferenceOrValue = Field(..., discriminator='discr', description="Either a ValueModel (direct filename) or ReferenceModel (reference to filename)")
    data: ReferenceOrValue = Field(..., discriminator='discr', description="Either a ValueModel (direct data) or ReferenceModel (reference to data)")
    result_reference_id: Optional[str] = Field(None, description="If provided, returns ReferenceModel for confirmation. If None, returns ValueModel.")

class WriteDataRefResponse(BaseModel):
    """Response wrapper for write_data_ref function."""
    result: ReferenceOrValue = Field(..., discriminator='discr', description="Either a ValueModel (if result_reference_id was None) or ReferenceModel (if result_reference_id was provided)")

# Request/Response wrapper models for retrieve_reference
class RetrieveReferenceRequest(BaseModel):
    """Request wrapper for retrieve_reference function."""
    reference_id: str = Field(..., description="The reference ID to retrieve")

class RetrieveReferenceResponse(BaseModel):
    """Response wrapper for retrieve_reference function."""
    result: ValueModel = Field(..., description="ValueModel containing the retrieved data")


## Tool Layer Implementation

The tool layer provides basic file operations using a shared dictionary. This is an implementation detail - the decorator layer will use these methods.


In [4]:
# Shared dictionary for tool layer (implementation detail)
_file_storage: dict[str, str] = {}

_large_file_content_prefix = "Large dataset content here... Very Big. Biggest file we have seen. Bigger than the whole of S3. Gigantic. "
_large_file_size = 10**6
_large_file_content = _large_file_content_prefix + ("B" * (_large_file_size - len(_large_file_content_prefix))) # 1MB of data
_large_file_name = "large_file.txt"

def initialize_file_storage():
    global _file_storage
    _file_storage.clear()
    _file_storage[_large_file_name] = _large_file_content
initialize_file_storage()


def get_data(filename: str) -> str:
    """Reads data from the file storage.
    
    Args:
        filename: The name of the file to read
        
    Returns:
        The data stored in the file
        
    Raises:
        KeyError: If the file does not exist
    """
    if filename not in _file_storage:
        raise KeyError(f"File '{filename}' does not exist")
    return _file_storage[filename]

def write_data(filename: str, data: str) -> str:
    """Writes data to the file storage.
    
    Args:
        filename: The name of the file to write
        data: The data to write (can overwrite existing files)
    """
    _file_storage[filename] = data
    return f"success writing file. File size: {len(data)} chars"


## Reference Store Implementation

The Reference Store maintains a registry of references separate from the tool layer's file storage. It prevents duplicate reference IDs and handles reference retrieval.


In [5]:
class ReferenceStore:
    """Manages references to data, separate from the tool layer's file storage."""
    
    def __init__(self):
        self._reference_registry: dict[str, str] = {}
    
    def store_reference(self, ref_id: str, data: str) -> None:
        """Stores data with a reference ID.
        
        Args:
            ref_id: The unique reference ID
            data: The data to store
            
        Raises:
            ValueError: If the reference ID already exists
        """
        if ref_id in self._reference_registry:
            raise ValueError(f"Reference ID '{ref_id}' already exists. Cannot set the same reference twice.")
        self._reference_registry[ref_id] = data
    
    def get_reference(self, ref_id: str) -> str:
        """Retrieves data by reference ID.
        
        Args:
            ref_id: The reference ID to retrieve
            
        Returns:
            The data associated with the reference ID
            
        Raises:
            KeyError: If the reference ID does not exist
        """
        if ref_id not in self._reference_registry:
            raise KeyError(f"Reference ID '{ref_id}' does not exist")
        return self._reference_registry[ref_id]
    
    def has_reference(self, ref_id: str) -> bool:
        """Checks if a reference ID exists."""
        return ref_id in self._reference_registry

# Global reference store instance
reference_store = ReferenceStore()


## Reference Store Layer Decorator Methods

These methods wrap the tool layer and support both value and reference modes. They will be exposed as tools to the LLM.


In [6]:
# Global flag to control logging (disabled by default, enabled in LLM integration)
print_logs_enabled = False

def print_logs(message: str):
    """Print log message only if logging is enabled."""
    if print_logs_enabled:
        print(message)

def _resolve_value(value_or_ref: ReferenceOrValue) -> tuple[str, str]:
    """Helper function to resolve a ReferenceOrValue to a string and return mode info.
    
    Args:
        value_or_ref: Either a ValueModel or ReferenceModel
        
    Returns:
        Tuple of (resolved_string, mode_description)
    """
    if isinstance(value_or_ref, ValueModel):
        return value_or_ref.data, "value"
    elif isinstance(value_or_ref, ReferenceModel):
        return reference_store.get_reference(value_or_ref.ref_id), f"reference(ref_id='{value_or_ref.ref_id}')"
    else:
        raise TypeError(f"Expected ValueModel or ReferenceModel, got {type(value_or_ref)}")

def _format_log_entry(method_name: str, inputs: dict, output_type: str, output_info: str) -> str:
    """Format a pretty log entry for decorator methods."""
    input_lines = []
    for k, v in inputs.items():
        input_lines.append(f"      {k}: {v}")
    input_str = "\n".join(input_lines)
    return f"{method_name}\n   Inputs:\n{input_str}\n   Output: {output_type}({output_info})"

def get_data_ref(request: GetDataRefRequest) -> GetDataRefResponse:
    """Reads data from file storage, supporting both value and reference modes.
    
    Args:
        request: GetDataRefRequest containing filename and optional result_reference_id
    
    Returns:
        GetDataRefResponse containing either a ValueModel (if result_reference_id is None) or ReferenceModel (if result_reference_id is provided)
    """
    filename = request.filename
    result_reference_id = request.result_reference_id
    
    # Determine input mode
    filename_mode = "value" if isinstance(filename, ValueModel) else f"reference(ref_id='{filename.ref_id}')"
    filename_value = filename.data if isinstance(filename, ValueModel) else filename.ref_id
    
    # Resolve filename (could be value or reference)
    resolved_filename, resolved_mode = _resolve_value(filename)
    
    # Call tool layer
    data = get_data(resolved_filename)
    data_preview = data[:50] + "..." if len(data) > 50 else data
    
    # Handle result_reference_id
    if result_reference_id is not None:
        # Store result as reference
        reference_store.store_reference(result_reference_id, data)
        output_type = "ReferenceModel"
        output_info = f"ref_id='{result_reference_id}' (stored {len(data)} chars)"
        result = ReferenceModel(ref_id=result_reference_id)
    else:
        # Return as value
        output_type = "ValueModel"
        output_info = f"data='{data_preview}' ({len(data)} chars)"
        result = ValueModel(data=data)
    
    # Log the operation
    inputs = {
        f"filename ({filename_mode})": f"'{filename_value}' → resolved to '{resolved_filename}'",
        "result_reference_id": f"'{result_reference_id}'" if result_reference_id else "None"
    }
    log_msg = _format_log_entry("🔍 get_data_ref", inputs, output_type, output_info)
    print_logs(f"\n{log_msg}\n")
    
    return GetDataRefResponse(result=result)

def write_data_ref(request: WriteDataRefRequest) -> WriteDataRefResponse:
    """Writes data to file storage, supporting both value and reference modes.
    
    Args:
        request: WriteDataRefRequest containing filename, data, and optional result_reference_id
    
    Returns:
        WriteDataRefResponse containing either a ValueModel (if result_reference_id is None) or ReferenceModel (if result_reference_id is provided)
    """
    filename = request.filename
    data = request.data
    result_reference_id = request.result_reference_id
    
    # Determine input modes
    filename_mode = "value" if isinstance(filename, ValueModel) else f"reference(ref_id='{filename.ref_id}')"
    filename_value = filename.data if isinstance(filename, ValueModel) else filename.ref_id
    
    data_mode = "value" if isinstance(data, ValueModel) else f"reference(ref_id='{data.ref_id}')"
    data_value = data.data if isinstance(data, ValueModel) else data.ref_id
    
    # Resolve both filename and data (could be values or references)
    resolved_filename, resolved_filename_mode = _resolve_value(filename)
    resolved_data, resolved_data_mode = _resolve_value(data)
    data_preview = resolved_data[:50] + "..." if len(resolved_data) > 50 else resolved_data
    
    # Call tool layer
    write_result = write_data(resolved_filename, resolved_data)
    
    # Handle result_reference_id
    if result_reference_id is not None:
        # Store written data as reference
        reference_store.store_reference(result_reference_id, write_result)
        output_type = "ReferenceModel"
        output_info = f"ref_id='{result_reference_id}' (stored {len(resolved_data)} chars)"
        result = ReferenceModel(ref_id=result_reference_id)
    else:
        output_type = "ValueModel"
        output_info = f"data='{write_result}'"
        result = ValueModel(data=write_result)
    
    # Log the operation
    inputs = {
        f"filename ({filename_mode})": f"'{filename_value}' → resolved to '{resolved_filename}'",
        f"data ({data_mode})": f"'{data_value}' → resolved to '{data_preview}' ({len(resolved_data)} chars)",
        "result_reference_id": f"'{result_reference_id}'" if result_reference_id else "None"
    }
    log_msg = _format_log_entry("✍️  write_data_ref", inputs, output_type, output_info)
    print_logs(f"\n{log_msg}\n")
    
    return WriteDataRefResponse(result=result)

def retrieve_reference(request: RetrieveReferenceRequest) -> RetrieveReferenceResponse:
    """Retrieves data from the reference store by reference ID.
    
    Args:
        request: RetrieveReferenceRequest containing the reference_id to retrieve
        
    Returns:
        RetrieveReferenceResponse containing ValueModel with the retrieved data
        
    Raises:
        KeyError: If the reference ID does not exist
    """
    reference_id = request.reference_id
    data = reference_store.get_reference(reference_id)
    data_preview = data[:50] + "..." if len(data) > 50 else data
    
    # Log the operation
    inputs = {
        "reference_id": f"'{reference_id}'"
    }
    output_type = "ValueModel"
    output_info = f"data='{data_preview}' ({len(data)} chars)"
    log_msg = _format_log_entry("🔎 retrieve_reference", inputs, output_type, output_info)
    print_logs(f"\n{log_msg}\n")
    
    return RetrieveReferenceResponse(result=ValueModel(data=data))


## Test Cases

Let's test the implementation with various scenarios to ensure everything works correctly.


In [7]:
# Reset state for testing
initialize_file_storage()
reference_store._reference_registry.clear()

print("=== Test 1: Value Mode - Basic Operations ===")
# Write using value mode
write_response = write_data_ref(
    WriteDataRefRequest(
        filename=ValueModel(data="test.txt"),
        data=ValueModel(data="Hello, World!")
    )
)
print("✓ Write with value mode successful")

# Read using value mode
result_response = get_data_ref(GetDataRefRequest(filename=ValueModel(data="test.txt")))
result = result_response.result
print(f"✓ Read with value mode: {result}")
assert isinstance(result, ValueModel)
assert result.data == "Hello, World!"
print("✓ Value mode test passed\n")


=== Test 1: Value Mode - Basic Operations ===
✓ Write with value mode successful
✓ Read with value mode: discr='value' data='Hello, World!'
✓ Value mode test passed



In [8]:
print("=== Test 2: Reference Mode - Using result_reference_id ===")
# Write data and store as reference
write_data_ref(
    WriteDataRefRequest(
        filename=ValueModel(data="data.txt"),
        data=ValueModel(data="Some data content here..."),
        result_reference_id="data_ref_1"
    )
)
print("✓ Write with result_reference_id successful")

# Retrieve the data reference
retrieved_data_response = get_data_ref(
    GetDataRefRequest(filename=ValueModel(data="data.txt"))
)
retrieved_data = retrieved_data_response.result
print(f"✓ Retrieved reference: {retrieved_data.data[:30]}...")
assert isinstance(retrieved_data, ValueModel)
assert retrieved_data.data == "Some data content here..."

# Retrieve the result reference
retrieved_result_response = retrieve_reference(RetrieveReferenceRequest(reference_id="data_ref_1"))
retrieved_result = retrieved_result_response.result
print(f"✓ Retrieved reference: {retrieved_result.data[:30]}...")
assert isinstance(retrieved_result, ValueModel)
assert retrieved_result.data.startswith("success writing file")
print("✓ Reference mode test passed\n")


=== Test 2: Reference Mode - Using result_reference_id ===
✓ Write with result_reference_id successful
✓ Retrieved reference: Some data content here......
✓ Retrieved reference: success writing file. File siz...
✓ Reference mode test passed



In [9]:
print("=== Test 3: Mixed Mode - Reference as Input ===")
# First, create a reference to a filename
reference_store.store_reference("filename_ref", "config.txt")
print("✓ Created filename reference")

# Write using reference for filename
write_data_ref(
    WriteDataRefRequest(
        filename=ReferenceModel(ref_id="filename_ref"),
        data=ValueModel(data="Configuration data"),
        result_reference_id="config_ref"
    )
)
print("✓ Write with reference filename successful")

# Read using reference for filename
result_response = get_data_ref(
    GetDataRefRequest(
        filename=ReferenceModel(ref_id="filename_ref"),
        result_reference_id="read_result_ref"
    )
)
result = result_response.result
print(f"✓ Read with reference filename: {result}")
assert isinstance(result, ReferenceModel)
assert result.ref_id == "read_result_ref"

# Verify the data was stored correctly
read_data_response = retrieve_reference(RetrieveReferenceRequest(reference_id="read_result_ref"))
read_data = read_data_response.result
assert read_data.data == "Configuration data"
print("✓ Mixed mode test passed\n")


=== Test 3: Mixed Mode - Reference as Input ===
✓ Created filename reference
✓ Write with reference filename successful
✓ Read with reference filename: discr='reference' ref_id='read_result_ref'
✓ Mixed mode test passed



In [10]:
print("=== Test 4: Error Cases ===")
# Test: Setting duplicate reference ID
try:
    reference_store.store_reference("duplicate_ref", "First data")
    reference_store.store_reference("duplicate_ref", "Second data")
    print("✗ Should have raised ValueError for duplicate reference")
    assert False
except ValueError as e:
    print(f"✓ Correctly raised ValueError: {e}")

# Test: Retrieving non-existent reference
try:
    retrieve_reference(RetrieveReferenceRequest(reference_id="nonexistent_ref"))
    print("✗ Should have raised KeyError for non-existent reference")
    assert False
except KeyError as e:
    print(f"✓ Correctly raised KeyError: {e}")

# Test: Reading non-existent file
try:
    get_data_ref(GetDataRefRequest(filename=ValueModel(data="nonexistent.txt")))
    print("✗ Should have raised KeyError for non-existent file")
    assert False
except KeyError as e:
    print(f"✓ Correctly raised KeyError: {e}")

print("✓ Error handling tests passed\n")


=== Test 4: Error Cases ===
✓ Correctly raised ValueError: Reference ID 'duplicate_ref' already exists. Cannot set the same reference twice.
✓ Correctly raised KeyError: "Reference ID 'nonexistent_ref' does not exist"
✓ Correctly raised KeyError: "File 'nonexistent.txt' does not exist"
✓ Error handling tests passed



In [11]:
print("=== Test 5: result_reference_id Behavior ===")
# Test: get_data_ref with result_reference_id
write_data("test2.txt", "Test content for reference")

# Without result_reference_id - returns ValueModel
result1_response = get_data_ref(GetDataRefRequest(filename=ValueModel(data="test2.txt")))
result1 = result1_response.result
assert isinstance(result1, ValueModel)
print(f"✓ Without result_reference_id: returns ValueModel")

# With result_reference_id - returns ReferenceModel
result2_response = get_data_ref(
    GetDataRefRequest(
        filename=ValueModel(data="test2.txt"),
        result_reference_id="test_result_ref"
    )
)
result2 = result2_response.result
assert isinstance(result2, ReferenceModel)
assert result2.ref_id == "test_result_ref"
print(f"✓ With result_reference_id: returns ReferenceModel")

# Verify the reference was stored
stored_data_response = retrieve_reference(RetrieveReferenceRequest(reference_id="test_result_ref"))
stored_data = stored_data_response.result
assert stored_data.data == "Test content for reference"
print("✓ Reference was correctly stored")
print("✓ result_reference_id behavior test passed\n")


=== Test 5: result_reference_id Behavior ===
✓ Without result_reference_id: returns ValueModel
✓ With result_reference_id: returns ReferenceModel
✓ Reference was correctly stored
✓ result_reference_id behavior test passed



In [12]:
print("=== Test 6: Large File Reference Roundtrip ===")

# 1. Create large content values
large_content = _large_file_content
large_filename = _large_file_name
large_filesize = _large_file_size
# _large_file_content_prefix


# 2. Read file contents into ValueModel using get_data_ref (no reference)
val_large_response = get_data_ref(GetDataRefRequest(filename=ValueModel(data=large_filename)))
val_large = val_large_response.result
assert isinstance(val_large, ValueModel)
assert val_large.data == large_content
print("✓ Large file read directly matches content")

# 3. Read file and store as reference
large_ref_id = "large_file_ref"
ref_large_response = get_data_ref(
    GetDataRefRequest(
        filename=ValueModel(data=large_filename),
        result_reference_id=large_ref_id,
    )
)
ref_large = ref_large_response.result
assert isinstance(ref_large, ReferenceModel)
assert ref_large.ref_id == large_ref_id

# 4. Retrieve the referenced content and confirm it matches
retrieved_response = retrieve_reference(RetrieveReferenceRequest(reference_id=large_ref_id))
retrieved = retrieved_response.result
assert retrieved.data == large_content
print("✓ Large file content stored in reference matches original")

# 5. Write reference content into a new, random file
random_file_name = "random_filename_123_xyz.txt"

# Write data using write_data_ref, passing the random_file as filename and ReferenceModel as data
write_result_response = write_data_ref(
    WriteDataRefRequest(
        filename=ValueModel(data=random_file_name),
        data=ReferenceModel(ref_id=large_ref_id)
    )
)
write_result = write_result_response.result
assert isinstance(write_result, ValueModel)

# 6. Read random file and verify it also matches
val_random_response = get_data_ref(GetDataRefRequest(filename=ValueModel(data=random_file_name)))
val_random = val_random_response.result
assert isinstance(val_random, ValueModel)
assert val_random.data == large_content
print(f"✓ Random file '{random_file_name}' written from reference, contents match")

print("✓ Large file reference and roundtrip tests passed\n")


=== Test 6: Large File Reference Roundtrip ===
✓ Large file read directly matches content
✓ Large file content stored in reference matches original
✓ Random file 'random_filename_123_xyz.txt' written from reference, contents match
✓ Large file reference and roundtrip tests passed



## LLM Integration

Now let's set up the tools to be used with Google Gemini. The decorator layer methods will be exposed as tools that the LLM can call.


In [13]:
import json
from google.genai import types
from google.genai.types import FunctionDeclaration
from IPython.display import display, Markdown

def print_token_usage(response):
    """Pretty print token usage metadata from a response."""
    if hasattr(response, 'usage_metadata') and response.usage_metadata:
        usage = response.usage_metadata
        print(f"\n📊 Token Usage:")
        print(f"   • Prompt tokens: {usage.prompt_token_count:,}")
        print(f"   • Candidates tokens: {usage.candidates_token_count:,}")
        print(f"   • Total tokens: {usage.total_token_count:,}")
    else:
        print("\n📊 Token Usage: Not available")

def print_registry_state():
    """Pretty print the current state of the reference registry."""
    print("\n" + "=" * 60)
    print("📦 Reference Registry State")
    print("=" * 60)
    if not reference_store._reference_registry:
        print("   (empty)")
    else:
        for ref_id, data in reference_store._reference_registry.items():
            data_preview = data[:100] + "..." if len(data) > 100 else data
            print(f"   • {ref_id}:")
            print(f"     └─ Data length: {len(data):,} chars")
            print(f"     └─ Preview: {data_preview}")
    print("=" * 60)

def send_and_display(user_message: str):
    """Send a message to the chat, display the response, and print token usage.
    
    Args:
        user_message: The message to send to the LLM
        
    Returns:
        The response object from the chat
    """
    print(f"\n> User: {user_message}")
    response = chat.send_message(user_message)
    display(Markdown(response.text))
    print_token_usage(response)
    return response


# Enable logging for LLM integration
print_logs_enabled = True

# Reset state for LLM demo
initialize_file_storage()
reference_store._reference_registry.clear()

# Expose decorator layer methods as tools
reference_system = [
    get_data_ref,
    write_data_ref,
    retrieve_reference,
]

model_name = "gemini-2.5-flash-lite"  # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash-lite-preview-09-2025", "gemini-2.5-flash", "gemini-2.5-flash-preview-09-2025", "gemini-2.5-pro"] {"allow-input":true}

system_instruction = """You are a helpful assistant that can read and write data using a reference system.

**Function Call Structure:**
All functions take wrapper request objects. For example:
- `get_data_ref` takes a `GetDataRefRequest` object with:
  - `filename`: A ValueModel or ReferenceModel (see below)
  - `result_reference_id` (optional): If provided, stores result as a reference
- `write_data_ref` takes a `WriteDataRefRequest` object with:
  - `filename`: A ValueModel or ReferenceModel
  - `data`: A ValueModel or ReferenceModel
  - `result_reference_id` (optional): If provided, stores result as a reference

**Data Modes:**
You can work with data in two modes:
1. Value mode: Use ValueModel with `{"discr": "value", "data": "..."}` 
2. Reference mode: Use ReferenceModel with `{"discr": "reference", "ref_id": "..."}`

**Response Structure:**
All functions return wrapper response objects with a `result` field containing either a ValueModel or ReferenceModel.

**Normal operation (dynamic choice):**
When calling a tool and you have a choice between calling using the value or existing reference of any parameter, use reference mode for that parameter with the existing reference.
When calling a tool and you only have the existing reference of a parameter, use reference mode for that parameter. Don't request for the value, except you are explicitly asked to.
When calling a tool and you only have the value of a parameter, use value mode for that parameter.

**Explicit "use references" mode:**
When explicitly instructed to "use only references", "use references only", "use references", or similar phrasing:
1. First, if you need data that is the output of a **NEW** tool call that has the result_reference_id parameter, call the tool with result_reference_id set to an autoincrementing ID in the form of `llm_ref_{n}` where n keeps increasing (e.g., "llm_ref_1", "llm_ref_2", etc.)
2. When you have a reference and need to pass it to another tool, ALWAYS pass it directly using ReferenceModel.

**Requesting data as reference:**
When you are explicitly asked to use references for data you do not currently have (but not in "use references" mode), request for the data using an autoincrementing reference ID in the form of `llm_ref_{n}` where n keeps increasing.

**Example Calls (JSON arguments you should send):**
1. Write a value directly:
   `write_data_ref({"filename": {"discr": "value", "data": "source.txt"}, "data": {"discr": "value", "data": "Hello"}})`
2. Read a file and store the result as a reference:
   `get_data_ref({"filename": {"discr": "value", "data": "source.txt"}, "result_reference_id": "llm_ref_1"})`
3. Write a value filename and a reference data directly:
   `write_data_ref({"filename": {"discr": "value", "data": "destination.txt"}, "data": {"discr": "reference", "ref_id": "llm_ref_1"}})`

Always explain what you're doing and whether you're using value or reference mode."""

chat = client.chats.create(
    model=model_name,
    config=types.GenerateContentConfig(
        tools=reference_system,
        system_instruction=system_instruction,
    ),
)

print("✓ Chat session created with reference system tools")
print("✓ Logging enabled for decorator methods")


✓ Chat session created with reference system tools
✓ Logging enabled for decorator methods


## Example Usage with LLM

Let's demonstrate how the LLM can use the reference system to manage data efficiently.


In [14]:

print("Example: LLM using reference system\n")
print("=" * 60)

# Example 1: Write data using value mode
send_and_display("Write 'Hello, World!' to a file called 'greeting.txt'")

# Example 2: Read data and store as reference
send_and_display("Read file greeting.txt into a result reference called 'greeting_ref'")

# Example 3: Copy data over to another file
send_and_display(f"Copy the contents of file '{_large_file_name}' into 'another_large_file.txt' using only references. Think step by step.")

# Example 4: Use the reference for original file
send_and_display("What data is stored in the reference 'greeting_ref'?")

print("\n" + "=" * 60)
print("\n✓ Example usage complete")

# Print final registry state
print_registry_state()


Example: LLM using reference system


> User: Write 'Hello, World!' to a file called 'greeting.txt'

✍️  write_data_ref
   Inputs:
      filename (value): 'greeting.txt' → resolved to 'greeting.txt'
      data (value): 'Hello, World!' → resolved to 'Hello, World!' (13 chars)
      result_reference_id: None
   Output: ValueModel(data='success writing file. File size: 13 chars')



I have written 'Hello, World!' to the file 'greeting.txt' in value mode.


📊 Token Usage:
   • Prompt tokens: 1,458
   • Candidates tokens: 20
   • Total tokens: 1,478

> User: Read file greeting.txt into a result reference called 'greeting_ref'

🔍 get_data_ref
   Inputs:
      filename (value): 'greeting.txt' → resolved to 'greeting.txt'
      result_reference_id: 'greeting_ref'
   Output: ReferenceModel(ref_id='greeting_ref' (stored 13 chars))



I have read the file 'greeting.txt' and stored it in a reference called 'greeting_ref'.


📊 Token Usage:
   • Prompt tokens: 1,575
   • Candidates tokens: 22
   • Total tokens: 1,597

> User: Copy the contents of file 'large_file.txt' into 'another_large_file.txt' using only references. Think step by step.

🔍 get_data_ref
   Inputs:
      filename (value): 'large_file.txt' → resolved to 'large_file.txt'
      result_reference_id: 'large_file_ref_1'
   Output: ReferenceModel(ref_id='large_file_ref_1' (stored 1000000 chars))


✍️  write_data_ref
   Inputs:
      filename (value): 'another_large_file.txt' → resolved to 'another_large_file.txt'
      data (reference(ref_id='large_file_ref_1')): 'large_file_ref_1' → resolved to 'Large dataset content here... Very Big. Biggest fi...' (1000000 chars)
      result_reference_id: None
   Output: ValueModel(data='success writing file. File size: 1000000 chars')



I have copied the contents of 'large_file.txt' to 'another_large_file.txt' using only references.


📊 Token Usage:
   • Prompt tokens: 1,894
   • Candidates tokens: 27
   • Total tokens: 1,921

> User: What data is stored in the reference 'greeting_ref'?

🔎 retrieve_reference
   Inputs:
      reference_id: 'greeting_ref'
   Output: ValueModel(data='Hello, World!' (13 chars))



The reference 'greeting_ref' contains the value 'Hello, World!'.


📊 Token Usage:
   • Prompt tokens: 1,989
   • Candidates tokens: 16
   • Total tokens: 2,005


✓ Example usage complete

📦 Reference Registry State
   • greeting_ref:
     └─ Data length: 13 chars
     └─ Preview: Hello, World!
   • large_file_ref_1:
     └─ Data length: 1,000,000 chars
     └─ Preview: Large dataset content here... Very Big. Biggest file we have seen. Bigger than the whole of S3. Giga...


## Summary

This notebook demonstrated:

1. **Pydantic Models**: `ValueModel` and `ReferenceModel` with Union types that translate to `anyOf` in Google SDK
2. **Tool Layer**: Basic `get_data` and `write_data` methods using a shared dictionary
3. **Reference Store**: Separate registry for managing references with duplicate prevention
4. **Decorator Layer**: Methods that support both value and reference modes:
   - `get_data_ref`: Read data with optional reference storage
   - `write_data_ref`: Write data with optional reference storage
   - `retrieve_reference`: Retrieve data from reference store
5. **LLM Integration**: Tools exposed to Google Gemini for function calling
6. **Key Features**:
   - `result_reference_id` parameter allows LLM to control return format
   - References reduce context window usage
   - Duplicate reference prevention
   - Proper error handling

The reference system enables efficient context management by allowing the LLM to choose when to pass data as values (small data) or references (large data), reducing token usage in the context window.
